# Fine-tune GPT-4o-mini to detect toxic comments
### Dataset: mteb/toxic_conversations_50k (HuggingFace)
### Task: Binary classification — "toxic" vs "clean"

In [ ]:
import os
import json
import pickle
import random
from dotenv import load_dotenv
from openai import OpenAI
from huggingface_hub import login
from datasets import load_dataset
import time

In [ ]:
TRAIN_CACHE = "toxic_train.pkl"
TEST_CACHE  = "toxic_test.pkl"

LABEL_MAP = {0: "clean", 1: "toxic"}

In [ ]:
load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN')

openai = OpenAI()
login(os.environ['HF_TOKEN'], add_to_git_credential=True)

In [ ]:
def load_and_clean(split, max_per_class=None):
    """Load a split and return balanced list of {text, label} dicts."""
    raw = load_dataset("mteb/toxic_conversations_50k", split=split)

    buckets = {"toxic": [], "clean": []}
    for row in raw:
        text = row["text"].strip()
        label = LABEL_MAP[row["label"]]
        if len(text) < 15:          # skip very short noise
            continue
        if len(text) > 800:         # cap length for token efficiency
            text = text[:800]
        buckets[label].append({"text": text, "label": label})

    # Balance classes
    if max_per_class:
        for key in buckets:
            random.shuffle(buckets[key])
            buckets[key] = buckets[key][:max_per_class]

    combined = buckets["toxic"] + buckets["clean"]
    random.shuffle(combined)
    return combined

In [ ]:
if os.path.exists(TRAIN_CACHE) and os.path.exists(TEST_CACHE):
    print("Loading from cache...")
    with open(TRAIN_CACHE, "rb") as f:
        train = pickle.load(f)
    with open(TEST_CACHE, "rb") as f:
        test = pickle.load(f)
else:
    print("Downloading dataset from HuggingFace...")
    train = load_and_clean("train", max_per_class=300)
    test  = load_and_clean("test",  max_per_class=150)

    with open(TRAIN_CACHE, "wb") as f:
        pickle.dump(train, f)
    with open(TEST_CACHE, "wb") as f:
        pickle.dump(test, f)

print(f"✓ Train: {len(train)} | Test: {len(test)}")

# Subsets for fine-tuning

fine_tune_train      = train[:300]
fine_tune_validation = train[300:370]

print(f"✓ Fine-tune train: {len(fine_tune_train)} | Validation: {len(fine_tune_validation)}")

In [ ]:
# Message formatters 

SYSTEM_PROMPT = (
    "You are a content moderation classifier. "
    "Given a comment, reply with exactly one word: 'toxic' or 'clean'. "
    "No explanation, no punctuation — just the single label."
)

def messages_for_training(item):
    """Full message including the ground-truth label (used for fine-tuning)."""
    return [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": item["text"]},
        {"role": "assistant", "content": item["label"]},
    ]

def messages_for_inference(item):
    """Message without the label (used at inference time)."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": item["text"]},
    ]

# Sanity-check both formats
print("\nTraining format sample:")
print(json.dumps(messages_for_training(train[0]), indent=2))

print("\nInference format sample (no label):")
print(json.dumps(messages_for_inference(train[0]), indent=2))

In [ ]:
# JSONL helpers 

def to_jsonl(data):
    lines = [json.dumps({"messages": messages_for_training(item)}) for item in data]
    return "\n".join(lines)

def write_and_upload(data, filename):
    """Write JSONL file and upload to OpenAI Files API."""
    with open(filename, "w", encoding="utf-8") as f:
        f.write(to_jsonl(data))
    with open(filename, "rb") as f:
        uploaded = openai.files.create(file=f, purpose="fine-tune")
    print(f"✓ Uploaded {filename} → file id: {uploaded.id}")
    return uploaded

#Upload datasets
train_file      = write_and_upload(fine_tune_train,      "toxic_train.jsonl")
validation_file = write_and_upload(fine_tune_validation, "toxic_validation.jsonl")

In [ ]:
class FineTuneJob:
    _instance = None
    _state_file = "finetune_state.json"

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._load()
        return cls._instance

    def _load(self):
        """Load state from disk if it exists."""
        if os.path.exists(self._state_file):
            with open(self._state_file, "r") as f:
                data = json.load(f)
            self.job_id = data.get("job_id")
            self.fine_tuned_model = data.get("fine_tuned_model")
            print(f"✓ Loaded state: job_id={self.job_id}, model={self.fine_tuned_model}")
        else:
            self.job_id = None
            self.fine_tuned_model = None

    def _save(self):
        """Persist state to disk."""
        with open(self._state_file, "w") as f:
            json.dump({"job_id": self.job_id, "fine_tuned_model": self.fine_tuned_model}, f, indent=2)

    def set_job(self, job_id):
        self.job_id = job_id
        self._save()
        print(f"✓ Saved job_id: {job_id}")

    def set_model(self, model_name):
        self.fine_tuned_model = model_name
        self._save()
        print(f"✓ Saved fine_tuned_model: {model_name}")

    def sync(self):
        """Pull latest status from OpenAI and update model name if training completed."""
        if not self.job_id:
            print("No job_id saved yet.")
            return
        j = openai.fine_tuning.jobs.retrieve(self.job_id)
        print(f"Status: {j.status}")
        if j.fine_tuned_model:
            self.set_model(j.fine_tuned_model)
        return j

    def __repr__(self):
        return f"FineTuneJob(job_id={self.job_id}, fine_tuned_model={self.fine_tuned_model})"
    
ftJob = FineTuneJob()

In [ ]:
job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4o-mini-2024-07-18",
    seed=7,
    hyperparameters={"n_epochs": 2},
    suffix="toxic_classifier",
)


print(f"\n✓ Fine-tuning job created")
print(f"  Job ID : {job.id}")
print(f"  Status : {job.status}")
print(f"  Model  : gpt-4o-mini-2024-07-18 → ft:...:toxic_classifier:...")

ftJob.set_job(job.id)

# Monitor job
while True:
    j = ftJob.sync()
    if j.status == "succeeded":
        print("Training complete!")
        break
    elif j.status == "failed":
        print("Training failed.")
        break
    print(f"Status: {j.status} — checking again in 60s...")
    time.sleep(60)

### Evaluation.
Run if training succeeds

In [ ]:
# Inference with fine-tuned model

def classify_comment(item, model_name):
    """Use the fine-tuned model to classify a comment as toxic or clean."""
    response = openai.chat.completions.create(
        model=model_name,
        messages=messages_for_inference(item),
        max_tokens=5,
        seed=7,
        temperature=0,      # deterministic for classification
    )
    return response.choices[0].message.content.strip().lower()


#  Evaluation

def evaluate(size=100):
    """
    Sync job status, then run the fine-tuned model on `size` test examples
    and report accuracy plus a breakdown by class (toxic / clean).
    """
    ftJob.sync()

    if not ftJob.fine_tuned_model:
        print("Model not ready yet — check back after training completes.")
        return

    sample = random.sample(test, min(size, len(test)))

    correct = 0
    results = {"toxic": {"correct": 0, "total": 0},
               "clean": {"correct": 0, "total": 0}}

    for i, item in enumerate(sample):
        prediction = classify_comment(item, ftJob.fine_tuned_model)
        ground_truth = item["label"]

        results[ground_truth]["total"] += 1
        if prediction == ground_truth:
            correct += 1
            results[ground_truth]["correct"] += 1

        if (i + 1) % 20 == 0:
            print(f"  [{i+1}/{size}] running accuracy: {correct/(i+1):.1%}")

    overall = correct / len(sample)
    print(f"\n── Evaluation Results ──────────────────────────")
    print(f"  Overall accuracy : {overall:.1%}  ({correct}/{len(sample)})")
    for label, r in results.items():
        acc = r["correct"] / r["total"] if r["total"] else 0
        print(f"  {label:6s} accuracy : {acc:.1%}  ({r['correct']}/{r['total']})")
    return overall



# Quick single prediction
sample = test[0]
print(f"Comment   : {sample['text']}")
print(f"True label: {sample['label']}")
print(f"Predicted : {classify_comment(sample, ftJob.fine_tuned_model)}")

# Full evaluation on 100 test examples
evaluate(size=100)